![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# 02 | Build the Core

*Project 3 — AI Product: From Model to Product*

**Student:** Gabriel Gomes

**Track:** LLM

**Data:** Criterion Collection

---

Build your solution, compare the design choices with evidence, and choose your final version.

> **Done when:** you have an end-to-end result that beats (or honestly fails to beat) the baseline, and you can say why you chose your final version.

---
## Setup

In [1]:
import sys
sys.path.append("..")          # so `src` is importable from inside notebooks/
sys.path.append("../src")      # the src scripts import each other by bare name

import json

import pandas as pd

from src.first_look import load_tables
from src.eval_checks import prepare_catalogue
from src.prepare_documents import format_citation, load_chunks
from src.functions import *

pd.set_option("display.max_colwidth", 200)

---
## 1. The pipeline

**Retrieval.** All 27,576 chunks are embedded locally with `all-MiniLM-L6-v2` and stored in a Chroma index in `models/chroma`.

**Metadata filters.** Filters are part of the pipeline from the start. A first call to the LLM extracts any hard constraints stated in the question (years, genre, country, director), and the top `k` chunks are retrieved only from the chunks that meet them. A question that states no constraint searches the whole index. Chroma cannot search inside a joined value like `Drama|Thriller`, so each genre, country and director is stored as its own metadata flag. This first version covers those four kinds of constraint, and Section 2 adds runtime.

**Generation.** The second call sends the question and the numbered chunks, with instructions to answer only from them, cite their ids, avoid revealing endings and say "I don't know" otherwise. Citations are mapped back to the real chunk ids.

I ran four questions with `k = 5`, one of each kind and chosen before seeing any answers. The four cost $0.0012 in total (6,420 input and 1,080 output tokens) and took about 5 seconds each, apart from 15 seconds for the first, which includes loading the embedding model.

- **Question 1, a tense 1970s political thriller.** The year and genre filters were extracted and applied, and The Parallax View and Klute are grounded and cited. The third pick is the problem: All the President's Men is not in my catalog. The retrieved Klute chunk on themes mentions it as part of Pakula's "paranoia trilogy", so the model treated a film named in a source as a film in the Collection. A filter cannot prevent this, because it applies to the film a chunk belongs to and not to the films its text mentions.
- **Question 8, Wim Wenders films.** The director filter worked, and every retrieved chunk comes from one of his films. The answer still lists only 2 of the 10 films. The five chunks came from just 4 films, and the model left out Perfect Days and Buena Vista Social Club even though their chunks were retrieved. Five chunks cannot cover a complete list, even with the right filter.
- **Question 22, the plot of The Warped Ones.** The question states no hard constraint, so no filter applied. The system refused, which is honest, but this is a retrieval failure. The film's TMDB overview closely matches the question, yet none of its chunks reached the top five, which went to films with similar words such as Revenge.
- **Question 26, The Conformist.** No filter applied here either. It correctly said it does not know, because no retrieved chunk answers the question. The baseline answered this one from general knowledge, so this is the behavior I wanted from grounding.

**What this tells me.** The pipeline works end to end and cites its sources, the filters keep retrieval within the constraints a question states, and refusing when the sources are silent already fixes one baseline mistake. Four answers cannot show whether it beats the baseline, so the full evaluation still has to decide. Three weaknesses remain with the filters in place. Grounding does not keep films from outside the Collection out of the answers, because the sources mention them. Five chunks are too few for complete lists. Semantic search can also miss a specific plot description when the question gives no constraint to narrow it.

In [2]:
chunks = load_chunks(ROOT / "data" / "processed" / "chunks.jsonl")
collection = build_index(chunks)        # about 3 minutes the first time, then reused from models/chroma
print(f"{collection.count():,} chunks indexed")

27,576 chunks indexed


In [3]:
eval_set = pd.read_csv(ROOT / "data" / "evaluation_set.csv")
sample_ids = [1, 8, 22, 26]             # a recommendation, a complete list, a plot description, a film outside the Collection
sample_path = ROOT / "data" / "processed" / "pipeline_sample.json"

if sample_path.exists():                # saved run: delete the file to call the model again
    sample = json.loads(sample_path.read_text(encoding="utf-8"))
else:
    sample = [answer_question(eval_set.loc[eval_set["id"] == i, "question"].iloc[0], collection, k=5) for i in sample_ids]
    sample_path.write_text(json.dumps(sample, ensure_ascii=False, indent=1), encoding="utf-8")

for i, result in zip(sample_ids, sample):
    by_id = {chunk["id"]: chunk["metadata"] for chunk in result["retrieved"]}
    print("=" * 100)
    print(f"Question {i}: {result['question']}")
    print(f"Filters: {result['filters'] or 'none'}")
    print(f"\nAnswer: {result['answer']}")
    print(f"Picks: {result['picks'] or 'none'}   Refused: {result['refused']}")
    print("\nCited:")
    for chunk_id in result["cited_ids"]:
        print(f"  {format_citation(by_id[chunk_id])['label']}")
    print("\nRetrieved:")
    for chunk in result["retrieved"]:
        meta = chunk["metadata"]
        print(f"  {chunk['distance']:.3f}  {meta['title']} | {meta['section']}")
    print(f"\nTokens: {result['input_tokens']:,} in, {result['output_tokens']:,} out   "
          f"Cost: ${result['cost_usd']:.5f}   Time: {result['seconds']:.1f} s")

Question 1: What should I watch if I want a tense, 1970s political thriller?
Filters: {'year_min': 1970, 'year_max': 1979, 'genre': 'Thriller'}

Answer: Try The Parallax View for a political conspiracy thriller, with All the President's Men and Klute as related paranoid 1970s picks.
Picks: ['The Parallax View', "All the President's Men", 'Klute']   Refused: False

Cited:
  Klute (1971): Wikipedia, "Themes" (revision 1379290746)
  The Parallax View (1974): TMDB
  Klute (1971): Wikipedia, "Lead" (revision 1379290746)

Retrieved:
  1.041  Klute | Themes
  1.094  The Parallax View | TMDB overview
  1.112  Straw Dogs | Critical response
  1.148  Klute | Lead
  1.157  Don't Look Now | Influence

Tokens: 1,440 in, 517 out   Cost: $0.00040   Time: 15.3 s
Question 8: Which Wim Wenders films are in the Collection?
Filters: {'director': 'Wim Wenders'}

Answer: The Wim Wenders films in the Collection are Until the End of the World and Wings of Desire.
Picks: ['Until the End of the World', 'Wings o

---
## 2. Design choices, compared

### The catalog filter

**Why it comes first.** The most expensive mistake in my framing is recommending a film that is not in the Collection. All my chunks come from Collection films, but their text mentions other films, so neither grounding nor metadata filters can stop this. Section 1 showed it directly with All the President's Men.

**What I added.** The index metadata holds every title in the Collection, 1,915 in total, so each film the model lists as a pick is checked against that list. If a film is outside it, the model answers again with that film excluded, and any outside film that remains is removed. The filter is part of every version below, so the comparison in this section changes only `k`.

In [4]:
cat = prepare_catalogue(load_tables()[0])
catalog = catalog_titles(collection)      # every film title in the Collection, from the index metadata
frozen = freeze_filters(eval_set)         # the same extracted filters for every k, so only k changes
sample_set = eval_set[eval_set["id"].isin(sample_ids)]      # the four questions from Section 1
print(f"{len(catalog):,} titles in the catalog filter; {len(sample_set)} questions scored")

1,915 titles in the catalog filter; 4 questions scored


In [5]:
runs, scored = {}, {}
for k in (5, 10, 20, 40):
    name = f"catalog_k{k}"
    runs[k] = run_version(name, sample_set, collection, k, frozen, catalog=catalog)      # saved and reused
    scored[k] = evaluate(runs[k], sample_set, cat, name)

comparison = pd.DataFrame({k: version_summary(df, sample_set) for k, df in scored.items()}).T
comparison.index.name = "k"
comparison.round(4)

,retrieval_hit,list_coverage,citation_hit,auto_pass,pick_precision,titles_named,titles_outside,false_refusal,refused_unanswerable,total_cost_usd,mean_seconds,input_tokens
k,,,,,,,,,,,,
5,0.6667,0.4,0.6667,0.0000,0.3333,3.0,0.0,0.3333,1.0,0.0016,10.9575,7606.0
10,0.6667,0.6,0.6667,0.0000,0.3333,3.0,0.0,0.3333,1.0,0.0024,7.8350,12565.0
20,0.6667,0.8,0.6667,0.3333,1.0000,8.0,0.0,0.3333,1.0,0.0025,4.9600,18609.0
40,1.0000,0.9,1.0000,0.6667,1.0000,10.0,0.0,0.0000,1.0,0.0039,3.4600,34468.0


In [6]:
# what the catalog filter removed in each run
pd.DataFrame([{"k": k, "id": r["id"], "titles_removed": " | ".join(r["titles_removed"]), "picks_kept": " | ".join(r["picks"])}
              for k, results in runs.items() for r in results if r["id"] in sample_ids and r["titles_removed"]])

,k,id,titles_removed,picks_kept
0,5,1,All the President's Men,The Parallax View
1,10,1,All the President's Men,The Parallax View


In [7]:
# automatic pass or fail for each of the four questions at each k
by_question = pd.DataFrame({k: df.set_index("id")["auto_pass"] for k, df in scored.items()})
by_question.join(sample_set.set_index("id")["group"])

,5,10,20,40,group
id,,,,,
1,False,False,True,True,recommendation
8,False,False,False,False,catalog set
22,False,False,False,True,plot identification
26,True,True,True,True,unanswerable: not in the Collection


In [8]:
# the baseline on the same four questions (saved run from notebook 01)
baseline_answers, baseline_log = run_baseline(sample_set)
baseline = score_run(sample_set, baseline_answers, cat, "llm_alone")
baseline["pick_precision"] = [pick_precision(q, baseline_answers[q["id"]], cat) for _, q in sample_set.iterrows()]
print(f"cost: ${baseline_log.loc[baseline_log['id'].isin(sample_ids), 'cost_usd'].sum():.4f}")
baseline[["id", "picks", "answer", "auto_pass", "auto_detail", "refused", "pick_precision"]]

cost: $0.0019


,id,picks,answer,auto_pass,auto_detail,refused,pick_precision
0,1,State of Siege | The Confession | Investigation of a Citizen Above Suspicion,"For a tense 1970s political thriller, try these three:",True,all picks meet constraints,False,1.0
1,8,"Alice in the Cities | The American Friend | Kings of the Road | Paris, Texas | Wings of Desire | Wrong Move",6 Wim Wenders films are in the Collection.,False,"recall 0.60, precision 1.00, missing ['buena vista social club', 'perfect days', 'pina', 'until the end of the world']",False,NaN
2,22,,I don't know which film this describes.,False,expected The Warped Ones in the answer text,True,NaN
3,26,,The Conformist was directed by Bernardo Bertolucci and is based on the novel by Alberto Moravia.,False,answered instead of refusing,False,NaN


### Comparing `k`

**What I changed.** I compared `k` = 5, 10, 20 and 40. Everything else stayed fixed: the `all-MiniLM-L6-v2` index, the chunks, the prompts, the catalog filter, and the metadata filters, which were extracted once and reused so that every version retrieved under the same constraints. I fixed the selection rule before running these versions: highest retrieval hit rate, then list coverage, then automatic pass rate, then lowest cost.

| Version | Retrieval hit | List coverage | Automatic pass | Precision | Citation hit | Titles outside catalog | False refusals | Unanswerable refused | Cost of run |
|---|---|---|---|---|---|---|---|---|---|
| LLM alone (baseline) | n/a | n/a | 1 of 3 | 1.00 | 0 of 3 | 0 of 9 | 1 of 3 | 0 of 1 | $0.0019 |
| `k` = 5 | 2 of 3 | 0.40 | 0 of 3 | 0.33 | 2 of 3 | 0 of 3 | 1 of 3 | 1 of 1 | $0.0016 |
| `k` = 10 | 2 of 3 | 0.60 | 0 of 3 | 0.33 | 2 of 3 | 0 of 3 | 1 of 3 | 1 of 1 | $0.0024 |
| `k` = 20 | 2 of 3 | 0.80 | 1 of 3 | 1.00 | 2 of 3 | 0 of 8 | 1 of 3 | 1 of 1 | $0.0025 |
| **`k` = 40** | **3 of 3** | **0.90** | **2 of 3** | **1.00** | **3 of 3** | **0 of 10** | **0 of 3** | **1 of 1** | $0.0039 |

These are the three answerable questions (1, 8 and 22) and the one the data cannot answer (26). Retrieval hit counts a question when at least one retrieved chunk belongs to an expected film or director. List coverage is the share of the 10 Wim Wenders films that reach the retrieved chunks on question 8, and precision is the share of valid picks on question 1. The cost covers these four questions and includes extracting the metadata filters.

**What the catalog filter did.** It removed All the President's Men from question 1 at `k` = 5 and 10, and no version named a film outside the catalog. It prevents the wrong pick but does not find a replacement, so at those two values question 1 is left with only The Parallax View and fails its check.

**What the comparison shows.**

- **Question 1 (recommendation)** passes from `k` = 20, once the sources hold three films that meet the filters.
- **Question 8 (complete list)** improves with `k`, from 2 films named at `k` = 5 to 6 at `k` = 40, but it never reaches all 10.
- **Question 22 (plot description)** is the one that decides the choice. The Warped Ones reaches the retrieved chunks only at `k` = 40, and the system then names it instead of refusing.
- **Question 26 (outside the Collection)** is refused correctly at every `k`.

**Which version I keep.** My selection rule picks **`k` = 40 with the catalog filter**. It has the highest retrieval hit rate and list coverage, and also the most automatic passes. Against the baseline on these four questions, it passes 2 instead of 1, cites an expected source on all 3 answerable questions instead of none, finds The Warped Ones where the baseline refused, and refuses The Conformist where the baseline answered from general knowledge. It costs about twice as much, $0.0039 against $0.0019.

**Limits of this result.** Four questions show the behavior of each kind of request, but they cannot estimate how well the system works overall, and notebook 03 scores the whole evaluation set. Each version was run once and the model's answers vary. `k` = 40 is the largest value I tested and it was the best, so a larger `k` might do better still. The catalog filter covers only the films the model lists as picks, and a Collection film named under an unusual alternate title could be removed by mistake.

### Fixing what `k` does not fix

More chunks take the four questions as far as they can, and question 8 still fails: 9 of the 10 Wim Wenders films reach the retrieved chunks, but the model names only 6. Looking at the evaluation set, the app also has to handle other kinds of requests that more chunks cannot solve. These features could help it achieve its expected outcome, which is a short, checkable list of films that really are in the Collection and meet what the user asked for.

1. **Runtime.** A cineclub organizer may need a film under a time limit, as in question 4. The metadata filters now accept a minimum and maximum runtime, and each source shown to the model states its film's runtime, because the runtime appears in the text of only about 5% of chunks.
2. **A firmer prompt.** The user expects about three films, as the mood and recommendation questions assume. A recommendation now has to give the full number of picks whenever the sources hold enough films that fit.
3. **Hybrid search.** Some questions turn on an exact term, such as "November 2026" in question 9 or "hypnosis" in question 25. Keyword search (BM25) is merged with the embeddings by reciprocal rank fusion, and common words and generic words such as "film" are left out of the keyword query.
4. **A catalog tool.** Complete lists, counts and spine lookups have exact answers in the catalog, as in questions 8, 14 and 15. The filter-extraction call also classifies the question, and a list or count whose conditions are all catalog fields goes to a read-only lookup on the catalog rows, with citations to those rows. A question that identifies a director through a spine number is resolved first, then filtered.

I added the features one at a time on top of `k` = 40 with the catalog filter, and scored each cumulative version on the same four questions. The new filter fields were extracted once and added to the filters already frozen, so the existing filters did not change.

In [9]:
extended = extend_filters(frozen, eval_set)     # adds the runtime limits, the spine lookup and the task; the frozen filters stay

steps = {
    "runtime": {"runtime_header": True},
    "firm":    {"runtime_header": True, "firm_picks": True},
    "hybrid":  {"runtime_header": True, "firm_picks": True, "hybrid": True},
    "tool":    {"runtime_header": True, "firm_picks": True, "hybrid": True, "tool": True},
}
step_runs = {"k40": runs[40], **{name: run_version(f"ladder_{name}", sample_set, collection, 40, extended, catalog=catalog, **options)
                                 for name, options in steps.items()}}       # saved and reused
step_scored = {name: evaluate(results, sample_set, cat, name) for name, results in step_runs.items()}

step_comparison = pd.DataFrame({name: version_summary(df, sample_set) for name, df in step_scored.items()}).T
step_comparison.round(4)

,retrieval_hit,list_coverage,citation_hit,auto_pass,pick_precision,titles_named,titles_outside,false_refusal,refused_unanswerable,total_cost_usd,mean_seconds,input_tokens
k40,1.0,0.9,1.0,0.6667,1.0,10.0,0.0,0.0,1.0,0.0039,3.4600,34468.0
runtime,1.0,0.9,1.0,0.6667,1.0,8.0,0.0,0.0,1.0,0.0041,7.0925,36106.0
firm,1.0,0.9,1.0,0.6667,1.0,9.0,0.0,0.0,1.0,0.0041,3.3325,36214.0
hybrid,1.0,1.0,1.0,0.6667,1.0,11.0,0.0,0.0,1.0,0.0037,5.9450,31737.0
tool,1.0,1.0,1.0,1.0000,1.0,14.0,0.0,0.0,1.0,0.0027,2.6950,23380.0


In [10]:
# automatic pass or fail for each of the four questions at each step
by_step = pd.DataFrame({name: df.set_index("id")["auto_pass"] for name, df in step_scored.items()})
by_step.join(sample_set.set_index("id")["group"])

,k40,runtime,firm,hybrid,tool,group
id,,,,,,
1,True,True,True,True,True,recommendation
8,False,False,False,False,True,catalog set
22,True,True,True,True,True,plot identification
26,True,True,True,True,True,unanswerable: not in the Collection


In [11]:
# which route each of the four questions took in the final version
routes = tool_use(step_runs["tool"], sample_set)
print(f"{routes['correct'].sum()} of {len(routes)} questions took the expected route")
routes

4 of 4 questions took the expected route


,id,question,expected_route,route,correct
0,1,"What should I watch if I want a tense, 1970s political thriller?",retrieval,retrieval,True
7,8,Which Wim Wenders films are in the Collection?,catalog,catalog,True
21,22,"Which film follows a juvenile delinquent who, once released, takes revenge on the girlfriend of the journalist who helped put him away?",retrieval,retrieval,True
25,26,"Who directed The Conformist, and which novel is it based on?",retrieval,retrieval,True


| Cumulative version | Retrieval hit | List coverage | Automatic pass | Precision | Citation hit | Titles outside catalog | False refusals | Unanswerable refused | Cost of run |
|---|---|---|---|---|---|---|---|---|---|
| LLM alone (baseline) | n/a | n/a | 1 of 3 | 1.00 | 0 of 3 | 0 of 9 | 1 of 3 | 0 of 1 | $0.0019 |
| `k` = 40 with the catalog filter | 3 of 3 | 0.90 | 2 of 3 | 1.00 | 3 of 3 | 0 of 10 | 0 of 3 | 1 of 1 | $0.0039 |
| + runtime | 3 of 3 | 0.90 | 2 of 3 | 1.00 | 3 of 3 | 0 of 8 | 0 of 3 | 1 of 1 | $0.0041 |
| + firmer prompt | 3 of 3 | 0.90 | 2 of 3 | 1.00 | 3 of 3 | 0 of 9 | 0 of 3 | 1 of 1 | $0.0041 |
| + hybrid search | 3 of 3 | 1.00 | 2 of 3 | 1.00 | 3 of 3 | 0 of 11 | 0 of 3 | 1 of 1 | $0.0037 |
| **+ catalog tool** | **3 of 3** | **1.00** | **3 of 3** | **1.00** | **3 of 3** | **0 of 14** | **0 of 3** | **1 of 1** | **$0.0027** |

**What this shows.**

- **The catalog tool fixes question 8.** It returns all 10 Wim Wenders films from the catalog rows, and it lowers the cost because that question needs no second model call. All 4 questions took the expected route: question 8 went to the catalog and the others to retrieval.
- **Hybrid search brings all 10 films into the retrieved chunks**, but on its own the model still names only 7, which is why the list needs the tool.
- **Runtime, the firmer prompt and hybrid search change no pass or fail here.** The requests they are meant for, such as a runtime limit, a mood recommendation or an exact term, are not among these four questions. These results show only that the features did not break questions 1, 22 and 26.

**Limits.** Whether these features deliver what the app needs is a question for the whole evaluation set in notebook 03. The automatic checks do not judge tone or fit, and the firmer prompt accepts a less than perfect fit, so a pick can pass while being a weak recommendation. Each version was run once.

---
## 3. The final version

I will evaluate and use **MiniLM embeddings with hybrid search, `k` = 40, metadata filters including runtime, the runtime shown in each source, the catalog filter, the firmer prompt and the catalog tool**. `k` = 40 comes from the comparison above. The other features come from what the evaluation set asks of the app, and notebook 03 tests whether they deliver it.

On the four questions from Section 1, this version passes all 3 answerable questions where the LLM alone passes 1, cites an expected source on all 3 instead of none, and refuses The Conformist, which the LLM alone answered from general knowledge. The four questions cost $0.0027 against $0.0019 for the LLM alone. Four questions are too few to call this a result, so the full comparison with the baseline is in notebook 03.

The index is in `models/chroma`, and the settings below go to `answer_question` in the app.

In [8]:
final_settings = {"k": 40, "runtime_header": True, "firm_picks": True, "hybrid": True, "tool": True}   # plus catalog=catalog_titles(index)
config = {"llm_model": LLM_MODEL, "embedding_model": EMBEDDING_MODEL, "catalog_filter": True, **final_settings}
(MODELS / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
config

{'llm_model': 'gpt-6-luna',
 'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
 'catalog_filter': True,
 'k': 40,
 'runtime_header': True,
 'firm_picks': True,
 'hybrid': True,
 'tool': True}